In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
print("\nTask 1: Converting numpy arrays to PyTorch tensors...")

X_train_tensor = torch.FloatTensor(X_train)
X_test_tensor = torch.FloatTensor(X_test)
y_train_tensor = torch.FloatTensor(y_train)
y_test_tensor = torch.FloatTensor(y_test)

print(f"X_train_tensor shape: {X_train_tensor.shape}")
print(f"X_test_tensor shape: {X_test_tensor.shape}")
print(f"y_train_tensor shape: {y_train_tensor.shape}")
print(f"y_test_tensor shape: {y_test_tensor.shape}")


In [ ]:
# 2. Create TensorDataset objects
print("\nTask 2: Creating TensorDataset objects...")

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")


In [ ]:
# 3. Create DataLoaders
print("\nTask 3: Creating DataLoaders with batch_size=32...")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Number of training batches: {len(train_loader)}")



In [ ]:
# 4. Print shape of one batch
print("\nTask 4: Inspecting one batch from train_loader...")

for batch_images, batch_labels in train_loader:
    print(f"Batch images shape: {batch_images.shape}")
    print(f"Batch labels shape: {batch_labels.shape}")
    print(f"Images dtype: {batch_images.dtype}")
    print(f"Labels dtype: {batch_labels.dtype}")
    break


        # Calculate loss
        loss = criterion(predictions, batch_labels)

        # Backward pass
        loss.backward()

        # Update weights
        optimizer.step()

        # Accumulate loss
        total_loss += loss.item()

    # Return average loss
    avg_loss = total_loss / len(train_loader)
    return avg_loss

print("Training loop function created!")


In [ ]:
# 5. Display sample images
print("\nTask 5: Displaying sample images...")

# Get one batch for visualization
sample_images, sample_labels = next(iter(train_loader))

# Display 8 sample images
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
axes = axes.ravel()

for i in range(8):
    # Convert from (C, H, W) to (H, W, C) for display
    img = sample_images[i].numpy().transpose(1, 2, 0)
    age = sample_labels[i].item()

    axes[i].imshow(img)
    axes[i].set_title(f'Age: {age:.0f}', fontsize=12, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Sample Training Images with Ages', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Task 1: Write your model class here:
print("\nTask 1: Creating model class with 4 linear layers...")

class AgePredictionModel(nn.Module):
    def __init__(self):
        super(AgePredictionModel, self).__init__()

        # Calculate input size: 3 channels * 36 height * 36 width
        input_size = 3 * 36 * 36

        # Define 4 linear layers with decreasing sizes
        self.fc1 = nn.Linear(input_size, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 1)  # Output: single age value

        # Activation function
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        # Flatten the input
        x = x.view(x.size(0), -1)

        # Pass through layers with ReLU activation
        x = self.relu(self.fc1(x))
        x = self.dropout(x)

        x = self.relu(self.fc2(x))
        x = self.dropout(x)

        x = self.relu(self.fc3(x))
        x = self.dropout(x)

        x = self.fc4(x)

        return x.squeeze()

print("Model architecture created successfully!")


In [ ]:
# Task 2: Write your training loop here:
print("\nTask 2: Creating training loop function...")

def train_one_epoch(model, train_loader, criterion, optimizer, device):
    """
    Train the model for one epoch

    Args:
        model: PyTorch model
        train_loader: DataLoader for training data
        criterion: Loss function
        optimizer: Optimizer
        device: Device to run on (CPU or GPU)

    Returns:
        Average training loss for the epoch
    """
    model.train()
    total_loss = 0.0

    for batch_images, batch_labels in train_loader:
        # Move data to device
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        predictions = model(batch_images)

        # Calculate loss
        loss = criterion(predictions, batch_labels)

        # Backward pass
        loss.backward()

        # Update weights
        optimizer.step()

        # Accumulate loss
        total_loss += loss.item()

    # Return average loss
    avg_loss = total_loss / len(train_loader)
    return avg_loss

print("Training loop function created!")


In [ ]:
# Task 3: Write your validation loop here:
print("\nTask 3: Creating validation loop function...")

def validate(model, test_loader, criterion, device):
    """
    Validate the model

    Args:
        model: PyTorch model
        test_loader: DataLoader for test data
        criterion: Loss function
        device: Device to run on (CPU or GPU)

    Returns:
        Average validation loss
    """
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch_images, batch_labels in test_loader:
            # Move data to device
            batch_images = batch_images.to(device)
            batch_labels = batch_labels.to(device)

            # Forward pass
            predictions = model(batch_images)

            # Calculate loss
            loss = criterion(predictions, batch_labels)

            # Accumulate loss
            total_loss += loss.item()

    # Return average loss
    avg_loss = total_loss / len(test_loader)
    return avg_loss

print("Validation loop function created!")


In [ ]:
# Task 4: Define device, model, loss, optimizer:
print("\nTask 4: Defining device, model, loss function, and optimizer...")

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize model
model = AgePredictionModel().to(device)
print(f"\nModel architecture:")
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Define loss function (MSE for regression)
criterion = nn.MSELoss()
print(f"\nLoss function: MSE (Mean Squared Error)")

# Define optimizer (Adam)
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(f"Optimizer: Adam with learning rate 0.001")

In [ ]:
# Task 5: Start training for 20 epochs:
print("\n" + "="*80)
print("Task 5: Training for 20 epochs...")
print("="*80)

num_epochs = 20
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    # Train
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)

    # Validate
    val_loss = validate(model, test_loader, criterion, device)
    val_losses.append(val_loss)

    # Print progress
    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f}")

print("\n" + "="*80)
print("Training completed!")
print("="*80)
print(f"Final Train Loss: {train_losses[-1]:.4f}")
print(f"Final Val Loss: {val_losses[-1]:.4f}")

In [ ]:
# Task 1: Write your code here:
print("\nTask 1: Plotting training and validation loss...")

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, 'b-o', label='Training Loss', linewidth=2)
plt.plot(range(1, num_epochs+1), val_losses, 'r-s', label='Validation Loss', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.title('Training and Validation Loss Over Epochs', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_losses, 'b-o', label='Training Loss', linewidth=2)
plt.plot(range(1, num_epochs+1), val_losses, 'r-s', label='Validation Loss', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.title('Training and Validation Loss (Log Scale)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.yscale('log')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here:
print("\nTask 2 (Bonus): Plotting predictions with actual images...")

# Get predictions on test set
model.eval()
sample_images, sample_labels = next(iter(test_loader))
sample_images = sample_images.to(device)

with torch.no_grad():
    predictions = model(sample_images)

# Move back to CPU for plotting
sample_images = sample_images.cpu()
sample_labels = sample_labels.cpu()
predictions = predictions.cpu()

# Display 12 samples with predictions
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.ravel()

for i in range(12):
    # Convert from (C, H, W) to (H, W, C) for display
    img = sample_images[i].numpy().transpose(1, 2, 0)
    actual_age = sample_labels[i].item()
    predicted_age = predictions[i].item()
    error = abs(actual_age - predicted_age)

    axes[i].imshow(img)
    axes[i].set_title(f'Actual: {actual_age:.0f} | Predicted: {predicted_age:.1f}\nError: {error:.1f}',
                     fontsize=11, fontweight='bold')
    axes[i].axis('off')

    # Color-code border based on error
    if error < 5:
        color = 'green'
    elif error < 10:
        color = 'orange'
    else:
        color = 'red'

    for spine in axes[i].spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(3)

plt.suptitle('Predictions on Test Set (Green: Error < 5, Orange: Error < 10, Red: Error ≥ 10)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Calculate and display overall metrics
print("\n" + "="*80)
print("OVERALL PERFORMANCE METRICS")
print("="*80)

all_predictions = []
all_labels = []

model.eval()
with torch.no_grad():
    for batch_images, batch_labels in test_loader:
        batch_images = batch_images.to(device)
        predictions = model(batch_images)
        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(batch_labels.numpy())

all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

mae = np.mean(np.abs(all_predictions - all_labels))
rmse = np.sqrt(np.mean((all_predictions - all_labels)**2))
mse = np.mean((all_predictions - all_labels)**2)

print(f"Mean Absolute Error (MAE): {mae:.2f} years")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} years")
print(f"Mean Squared Error (MSE): {mse:.2f}")

# Plot prediction scatter
plt.figure(figsize=(10, 8))
plt.scatter(all_labels, all_predictions, alpha=0.5, s=30, color='steelblue')
plt.plot([all_labels.min(), all_labels.max()],
         [all_labels.min(), all_labels.max()],
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Age', fontsize=12)
plt.ylabel('Predicted Age', fontsize=12)
plt.title('Actual vs Predicted Age (All Test Samples)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)